# **LINEART RESTORATION (CLOSER)**


# ***DATA***

In [ ]:
# CODE BLOCK 1 (LOAD DATASET)
# Downloads the zip built by the dataset generator notebook, unzips it, and
# restores the paths, file lists and IMAGE_SIZE from split.json.
import os, json, shutil, glob, time, subprocess

UPLOAD_MODE = "drive"     # "drive" (recommended for multi-GB zips) or "upload" (browser upload)
LOCAL_ZIP   = "/content/inkrepair_dataset.zip"
split_root  = "/content/split"

# 1. get the zip onto the local Colab disk
t0 = time.perf_counter()
if UPLOAD_MODE == "drive":
    import gdown
    gdown.download("https://drive.google.com/file/d/1NO8i_J5mSlh7YYuwh2ZyFqkIW7ISrGuk/view?usp=drive_link", LOCAL_ZIP, quiet=False, fuzzy=True)
else:
    from google.colab import files
    uploaded = files.upload()                      # pick the zip in the file dialog
    assert uploaded, "Nothing uploaded"
    name = next(iter(uploaded))
    src = os.path.abspath(name)                    # files.upload saves to the cwd
    if src != LOCAL_ZIP:
        os.replace(src, LOCAL_ZIP)

print(f"zip ready: {os.path.getsize(LOCAL_ZIP) / 1e9:.2f} GB "
      f"in {time.perf_counter() - t0:.0f} s")

# 2. unzip and locate the split folder (whatever the zip's top folder is)
tmp_dir = "/content/_unzip"
shutil.rmtree(tmp_dir, ignore_errors=True)
shutil.rmtree(split_root, ignore_errors=True)
t0 = time.perf_counter()
subprocess.run(["unzip", "-q", LOCAL_ZIP, "-d", tmp_dir], check=True)

found = glob.glob(os.path.join(tmp_dir, "**", "split.json"), recursive=True)
assert found, "split.json not found in the zip. Was it built by the dataset generator notebook?"
shutil.move(os.path.dirname(found[0]), split_root)
shutil.rmtree(tmp_dir, ignore_errors=True)
os.remove(LOCAL_ZIP)                               # free disk space
print(f"unzipped in {time.perf_counter() - t0:.0f} s -> {split_root}")

# 3. restore the split manifest
with open(os.path.join(split_root, "split.json")) as fh:
    manifest = json.load(fh)

IMAGE_SIZE        = manifest["image_size"]
train_files       = manifest["train"]
test_files        = manifest["test"]
sample_files      = manifest["sample"]

test_clean_dir = os.path.join(split_root, "clean_test")
test_corr_dir  = os.path.join(split_root, "corrupted_test")
pre_root       = os.path.join(split_root, "precorrupted")
pre_clean      = os.path.join(pre_root, "clean")
pre_corr       = os.path.join(pre_root, "corrupted")

# clean_training/ held symlinks and was excluded from the zip. The same images,
# already resized to IMAGE_SIZE, are in precorrupted/clean, so use that folder.
train_dir = pre_clean

# 4. sanity checks
for d in (test_clean_dir, test_corr_dir, pre_clean, pre_corr):
    assert os.path.isdir(d), f"Missing folder: {d}"

n_test_clean = len(os.listdir(test_clean_dir))
n_test_corr  = len(os.listdir(test_corr_dir))
assert n_test_clean == n_test_corr == len(test_files), \
    f"test pairs mismatch: {n_test_clean} clean / {n_test_corr} corrupted / {len(test_files)} listed"
assert all(os.path.exists(os.path.join(test_clean_dir, f)) for f in sample_files), \
    "preview files missing from clean_test"

pre_manifest_path = os.path.join(pre_root, "manifest.json")
if os.path.exists(pre_manifest_path):
    with open(pre_manifest_path) as fh:
        pm = json.load(fh)
    if not pm.get("complete", False):
        print("WARNING: pre-corrupted generation was not marked complete")
    print(f"pre-corrupted: {pm.get('n_variants')} variants/image (as configured)")

n_pre_clean = len([f for f in os.listdir(pre_clean) if f.endswith(".png")])
n_pre_corr  = len([f for f in os.listdir(pre_corr)
                   if f.endswith(".png") and not f.endswith(".tmp.png")])
print(f"IMAGE_SIZE {IMAGE_SIZE} | train {len(train_files)} "
      f"(pre-clean {n_pre_clean}, pre-corrupted pairs {n_pre_corr}) | "
      f"test {len(test_files)} | sample {sample_files}")
if n_pre_clean < len(train_files):
    print(f"WARNING: {len(train_files) - n_pre_clean} training images missing from precorrupted/clean")


# ***DATA LOADER (split)***

In [ ]:
# CODE BLOCK 4 (DATA LOADERS: pre-corrupted train + frozen test)
# Training pairs come from disk only (no on-the-fly corruption).
# Each epoch shows every training image once with a random stored variant,
# then applies the same paired augmentation to both images. Corruption is done
# on the crisp image before augmentation, so holes stay softened like the strokes.
import os, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms


class PrecorruptedPairDataset(Dataset):
    """Training set: pairs read from disk (random variant per load) + paired augmentation."""
    def __init__(self, clean_dir, corrupted_dir, files, image_size=IMAGE_SIZE, augment=True):
        self.clean_dir     = clean_dir
        self.corrupted_dir = corrupted_dir
        self.image_size    = image_size
        self.augment       = augment

        variants = {}
        for f in os.listdir(corrupted_dir):
            if f.endswith(".png") and not f.endswith(".tmp.png") and "_v" in f:
                variants.setdefault(f.rsplit("_v", 1)[0], []).append(f)

        stems = sorted(os.path.splitext(f)[0] for f in files)
        self.items = [(s, sorted(variants[s])) for s in stems
                      if s in variants
                      and os.path.exists(os.path.join(clean_dir, s + ".png"))]
        self.files = [s + ".png" for s, _ in self.items]

        n_var = [len(v) for _, v in self.items]
        print(f"[precorrupted] {len(self.items)} images, "
              f"{sum(n_var)} pairs ({min(n_var, default=0)}-{max(n_var, default=0)} variants/image)")
        if len(self.items) < len(stems):
            print(f"  WARNING: {len(stems) - len(self.items)} training images have no "
                  f"pre-corrupted pairs (generation incomplete?) and are skipped")

        self.to_tensor = transforms.ToTensor()
        # fill=1.0 = white on [0, 1] tensors
        self.augmentation = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.2),
            transforms.RandomRotation(
                degrees=10,
                interpolation=transforms.InterpolationMode.BILINEAR,
                fill=1.0),
            transforms.RandomAffine(
                degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05),
                interpolation=transforms.InterpolationMode.BILINEAR,
                fill=1.0),
        ])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        stem, vs = self.items[idx]
        v = vs[int(torch.randint(len(vs), (1,)))]     # torch RNG is seeded per worker
        clean = Image.open(os.path.join(self.clean_dir, stem + ".png")).convert("L")
        corr  = Image.open(os.path.join(self.corrupted_dir, v)).convert("L")
        assert clean.size == corr.size == (self.image_size, self.image_size), \
            f"{v}: expected {self.image_size}², got {clean.size} / {corr.size}"

        corrupted, clean = self.to_tensor(corr), self.to_tensor(clean)
        if self.augment:
            # one call on a 2-channel stack -> identical geometry for both images
            pair = self.augmentation(torch.cat([corrupted, clean], dim=0))
            corrupted, clean = pair[0:1], pair[1:2]
        return corrupted, clean


class FrozenPairDataset(Dataset):
    """Test set: pairs read straight from disk. No augmentation."""
    def __init__(self, clean_dir, corrupted_dir, files=None, image_size=IMAGE_SIZE):
        self.clean_dir     = clean_dir
        self.corrupted_dir = corrupted_dir
        self.files = sorted(files) if files is not None else sorted(
            f for f in os.listdir(clean_dir) if f.lower().endswith(".png"))
        missing = [f for f in self.files
                   if not os.path.exists(os.path.join(corrupted_dir, f))]
        assert not missing, f"Missing corrupted counterparts: {missing[:5]}"
        self.to_tensor  = transforms.ToTensor()
        self.image_size = image_size
        print(f"[frozen] {len(self.files)} pairs in {clean_dir}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        f = self.files[idx]
        clean = Image.open(os.path.join(self.clean_dir, f)).convert("L")
        corr  = Image.open(os.path.join(self.corrupted_dir, f)).convert("L")
        assert clean.size == corr.size == (self.image_size, self.image_size), \
            f"{f}: expected {self.image_size}², got {clean.size} / {corr.size}"
        return self.to_tensor(corr), self.to_tensor(clean), f


# datasets
pre_dataset    = PrecorruptedPairDataset(pre_clean, pre_corr, train_files)
train_dataset  = pre_dataset                     # alias used by CODE BLOCK 5
test_dataset   = FrozenPairDataset(test_clean_dir, test_corr_dir)
sample_dataset = FrozenPairDataset(test_clean_dir, test_corr_dir, files=sample_files)

# loaders
train_loader = DataLoader(pre_dataset, batch_size=4, shuffle=True,
                          num_workers=min(4, os.cpu_count()), pin_memory=True,
                          persistent_workers=True, drop_last=False)
test_loader  = DataLoader(test_dataset, batch_size=4, shuffle=False,
                          num_workers=2, pin_memory=True)
print("train_loader:", len(train_loader), "batches/epoch")

# the 4 fixed preview images, loaded once as tensors
ex_corrupted  = torch.stack([sample_dataset[i][0] for i in range(len(sample_dataset))])
ex_clean      = torch.stack([sample_dataset[i][1] for i in range(len(sample_dataset))])
EXAMPLE_FILES = sample_dataset.files
print("Preview tensors:", ex_corrupted.shape)

In [ ]:
# CODE BLOCK 5 (CHECK DATA LOADERS)

import torch
import numpy as np
import matplotlib.pyplot as plt

# 0. SPLIT INTEGRITY
print(f"train  : {len(train_dataset):5d} images")
print(f"test   : {len(test_dataset):5d} pairs")
print(f"sample : {len(sample_dataset):5d} pairs -> {EXAMPLE_FILES}")

train_names = set(train_dataset.files)
test_names  = set(test_dataset.files)
print("train/test overlap:", len(train_names & test_names))     # must be 0
print("sample inside test:", set(sample_dataset.files) <= test_names)

# 1. SHAPES / DTYPES / RANGE
corrupted_batch, clean_batch = next(iter(train_loader))
print("\n[train] corrupted:", tuple(corrupted_batch.shape), corrupted_batch.dtype)
print("[train] clean    :", tuple(clean_batch.shape), clean_batch.dtype)

t_corr, t_clean, t_names = next(iter(test_loader))
print("[test ] corrupted:", tuple(t_corr.shape), "| files:", list(t_names))

assert corrupted_batch.shape[1] == 1, "Expected 1 channel"
assert corrupted_batch.shape[-2:] == (IMAGE_SIZE, IMAGE_SIZE)
assert corrupted_batch.shape == clean_batch.shape
assert 0.0 <= corrupted_batch.min() and corrupted_batch.max() <= 1.0

def print_stats(name, img):
    print(f"{name:20s} min={img.min():.3f} max={img.max():.3f} "
          f"mean={img.mean():.3f} std={img.std():.3f} "
          f"ink_frac={(img < 0.5).float().mean():.4f}")

print()
print_stats("train clean",     clean_batch[0])
print_stats("train corrupted", corrupted_batch[0])
print_stats("test clean",      t_clean[0])
print_stats("test corrupted",  t_corr[0])

# Anti-aliased greys should be present (binarization happens only at inference)
u = torch.unique((clean_batch[0] * 255).round().int())
print(f"\nclean unique values: {len(u)} (first 20) -> {u[:20].tolist()}")

# 2. PAIR CONSISTENCY
# Corruption only paints WHITE over ink -> corrupted >= clean everywhere.
# A violation means the pair is misaligned (e.g. augmentation applied twice).
def pair_violations(corr, clean, tol=1/255):
    return (corr < clean - tol).float().mean().item()

print(f"\n[train] pixels darker in corrupted than clean: {pair_violations(corrupted_batch, clean_batch):.6f}")
print(f"[test ] pixels darker in corrupted than clean: {pair_violations(t_corr, t_clean):.6f}")

def hole_frac(corr, clean):
    """Same definition as LineLoss._hole_mask: erased = white in input, ink in target."""
    return ((corr > 0.9) & (clean < 0.5)).float().mean(dim=(1, 2, 3))

print("[train] hole_frac per image:", [f"{v:.4f}" for v in hole_frac(corrupted_batch, clean_batch)])
print("[test ] hole_frac per image:", [f"{v:.4f}" for v in hole_frac(t_corr, t_clean)])

# 3. TRAIN vs TEST DISTRIBUTION
# If these diverge, the frozen test set is not reproducing the training
# corruption and the test metrics will be biased.
@torch.no_grad()
def collect(loader, n_batches):
    holes, inks = [], []
    for k, batch in enumerate(loader):
        corr, clean = batch[0], batch[1]
        holes.append(hole_frac(corr, clean))
        inks.append((clean < 0.5).float().mean(dim=(1, 2, 3)))
        if k + 1 >= n_batches:
            break
    return torch.cat(holes).numpy(), torch.cat(inks).numpy()

train_holes, train_ink = collect(train_loader, 25)
test_holes,  test_ink  = collect(test_loader, 25)

print(f"\nhole_frac  train {train_holes.mean():.4f} ± {train_holes.std():.4f}"
      f"   |   test {test_holes.mean():.4f} ± {test_holes.std():.4f}")
print(f"ink_frac   train {train_ink.mean():.4f} ± {train_ink.std():.4f}"
      f"   |   test {test_ink.mean():.4f} ± {test_ink.std():.4f}")
print(f"empty-hole images in train sample: {(train_holes < 1e-6).sum()} / {len(train_holes)}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
bins = np.linspace(0, max(train_holes.max(), test_holes.max()) * 1.05, 30)
ax[0].hist(train_holes, bins=bins, alpha=.6, label="train", density=True)
ax[0].hist(test_holes,  bins=bins, alpha=.6, label="test",  density=True)
ax[0].set_title("hole fraction"); ax[0].legend()
ax[1].hist(train_ink, bins=30, alpha=.6, label="train", density=True)
ax[1].hist(test_ink,  bins=30, alpha=.6, label="test",  density=True)
ax[1].set_title("ink fraction"); ax[1].legend()
plt.tight_layout(); plt.show()

# 4. VISUAL CHECK - the 4 fixed preview images
def show_gray(ax, tensor, title):
    ax.imshow(tensor[0].numpy(), cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(title, fontsize=9); ax.axis("off")

n = len(sample_dataset)
fig, axes = plt.subplots(n, 3, figsize=(13, 4.4 * n))
for r in range(n):
    corr, clean = ex_corrupted[r], ex_clean[r]
    holes = ((corr > 0.9) & (clean < 0.5)).float()
    show_gray(axes[r, 0], clean, f"{EXAMPLE_FILES[r]} - clean")
    show_gray(axes[r, 1], corr,  "corrupted (frozen)")
    ov = np.repeat(clean[0].numpy()[..., None], 3, axis=2)
    ov[holes[0].numpy() > 0] = (1.0, 0.0, 0.0)          # red = erased ink
    axes[r, 2].imshow(ov, interpolation="nearest")
    axes[r, 2].set_title(f"holes ({holes.mean():.3%} of pixels)", fontsize=9)
    axes[r, 2].axis("off")
plt.tight_layout(); plt.show()

# Zoom on one hole region so you can see the anti-aliased edges
corr, clean = ex_corrupted[0], ex_clean[0]
ys, xs = torch.nonzero(((corr > 0.9) & (clean < 0.5))[0], as_tuple=True)
if len(ys):
    cy, cx = int(ys[len(ys)//2]), int(xs[len(xs)//2])
    y0, x0 = max(0, cy - 64), max(0, cx - 64)
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    show_gray(ax[0], clean[:, y0:y0+128, x0:x0+128], "clean (zoom)")
    show_gray(ax[1], corr[:,  y0:y0+128, x0:x0+128], "corrupted (zoom)")
    plt.tight_layout(); plt.show()

# **MODEL**

- 1 channel input/output
- Resolution: 896x896
- UNet + Attention
- The weight of the masked loss is important:
  - Too high: holes filled but bad the rest
  - Too low: holes not filled


In [ ]:
#CODE BLOCK 6 (MODEL/Sinusoidal embedding)

import torch
import torch.nn as nn
class SinusoidalPosEmbed2D(nn.Module):
    def __init__(self, dim, h, w):
        super().__init__()
        assert dim % 4 == 0, "dim must be divisible by 4"
        pe_h = self._make_pe(h, dim // 2)   # (H, dim/2)
        pe_w = self._make_pe(w, dim // 2)   # (W, dim/2)
        pe = torch.cat([pe_h.unsqueeze(1).expand(-1, w, -1),
                        pe_w.unsqueeze(0).expand(h, -1, -1)], dim=-1)  # (H, W, dim)
        self.register_buffer('pe', pe.reshape(1, h * w, dim))

    def _make_pe(self, length, dim):
        pos = torch.arange(length).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, dim, 2).float() * -(np.log(10000.0) / dim))
        pe = torch.zeros(length, dim)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe

    def forward(self, x):
        return x + self.pe

In [ ]:
#CODE BLOCK 7 (MODEL/Attention)
import torch
import torch.nn as nn

class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=8, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(dim * mlp_ratio), dim),
        )

    def forward(self, x):
        normed = self.norm1(x)
        x = x + self.attn(normed, normed, normed, need_weights=False)[0]   # no full attention matrix
        x = x + self.mlp(self.norm2(x))
        return x


class BottleneckAttention(nn.Module):
    def __init__(self, dim=512, heads=8, num_blocks=6, spatial_size=28):
        super().__init__()
        self.pos_embed = SinusoidalPosEmbed2D(dim, spatial_size, spatial_size)
        self.blocks = nn.Sequential(
            *[TransformerBlock(dim, heads) for _ in range(num_blocks)]
        )
        self.norm_out = nn.LayerNorm(dim)                     # rescales output before the decoder

    def forward(self, x):
        B, C, H, W = x.shape
        with torch.autocast('cuda', enabled=False):          # everything in here runs in float32
            tokens = x.float().flatten(2).permute(0, 2, 1)
            tokens = self.pos_embed(tokens)
            tokens = self.blocks(tokens)
            tokens = self.norm_out(tokens)                    # apply the exit normalization
            return tokens.permute(0, 2, 1).reshape(B, C, H, W)

In [ ]:
#CODE BLOCK 8 (MODEL/DilatedResBlock)

class DilatedResBlock(nn.Module):
    """Single residual block with one dilated 3x3 conv."""
    def __init__(self, channels, dilation):
        super().__init__()
        self.conv1 = nn.Conv2d(
            channels, channels, kernel_size=3,
            padding=dilation, dilation=dilation
        )
        self.norm1 = gn(channels)
        self.conv2 = nn.Conv2d(
            channels, channels, kernel_size=3,
            padding=1, dilation=1
        )
        self.norm2 = gn(channels)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.act(self.norm1(self.conv1(x)))
        out = self.norm2(self.conv2(out))
        return self.act(out + residual)


class DilatedContextStack(nn.Module):
    """
    Stack of residual blocks with increasing dilation, for long-range context.
    Designed to be applied at a mid-resolution feature map (e.g., 56x56).
    """
    def __init__(self, channels, dilations=(1, 2, 4, 8, 16)):
        super().__init__()
        self.blocks = nn.Sequential(
            *[DilatedResBlock(channels, d) for d in dilations]
        )

    def forward(self, x):
        return self.blocks(x)

In [ ]:
#CODE BLOCK 9 (MODEL/UNet)

import torch
import torch.nn as nn

def gn(num_channels, num_groups=8):
    """GroupNorm with safe group count (must divide num_channels)."""
    # Find largest valid group count <= num_groups
    while num_channels % num_groups != 0:
        num_groups -= 1
    return nn.GroupNorm(num_groups, num_channels)

class GatedConv2d(nn.Module):
    """
    Gated convolution: replaces nn.Conv2d in inpainting-style problems.
    Learns a soft, per-pixel mask of where the input is reliable.
    """
    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0):
        super().__init__()
        self.feature = nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding)
        self.gate    = nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding)

    def forward(self, x):
        feat = self.feature(x)
        mask = torch.sigmoid(self.gate(x))
        return feat * mask

class UNetEncoder(nn.Module):
    def __init__(self, c0=16):
        super().__init__()

        # Full-resolution stage (896x896), no downsampling
        self.enc0 = nn.Sequential(
            GatedConv2d(1, c0, kernel_size=3, stride=1, padding=1),
            gn(c0),
            nn.ReLU(),
            GatedConv2d(c0, c0, kernel_size=3, stride=1, padding=1),
            gn(c0),
            nn.ReLU(),
        )

        self.enc1 = nn.Sequential(
            GatedConv2d(c0, 32, kernel_size=4, stride=2, padding=1),   # in_ch = c0
            gn(32),
            nn.ReLU(),
            GatedConv2d(32, 32, kernel_size=3, stride=1, padding=1),
            gn(32),
            nn.ReLU(),
        )

        self.enc2 = nn.Sequential(
            GatedConv2d(32, 64, kernel_size=4, stride=2, padding=1),
            gn(64),
            nn.ReLU(),
            GatedConv2d(64, 64, kernel_size=3, stride=1, padding=1),
            gn(64),
            nn.ReLU(),
        )

        self.enc3 = nn.Sequential(
            GatedConv2d(64, 128, kernel_size=4, stride=2, padding=1),
            gn(128),
            nn.ReLU(),
            GatedConv2d(128, 128, kernel_size=3, stride=1, padding=1),
            gn(128),
            nn.ReLU(),
        )

        self.enc4 = nn.Sequential(
            GatedConv2d(128, 256, kernel_size=4, stride=2, padding=1),
            gn(256),
            nn.ReLU(),
            GatedConv2d(256, 256, kernel_size=3, stride=1, padding=1),
            gn(256),
            nn.ReLU(),
        )

        self.enc5 = nn.Sequential(
            GatedConv2d(256, 512, kernel_size=4, stride=2, padding=1),
            gn(512),
            nn.ReLU(),
            GatedConv2d(512, 512, kernel_size=3, stride=1, padding=1),
            gn(512),
            nn.ReLU(),
        )

    def forward(self, x):
        e0 = self.enc0(x)        # 896
        e1 = self.enc1(e0)       # 448
        e2 = self.enc2(e1)       # 224
        e3 = self.enc3(e2)       # 112
        e4 = self.enc4(e3)       # 56
        e5 = self.enc5(e4)       # 28
        return e5, [e0, e1, e2, e3, e4]


class UNetDecoder(nn.Module):
    def __init__(self, c0=16):
        super().__init__()

        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            gn(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            gn(256),
            nn.ReLU(),
        )

        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(512, 128, kernel_size=4, stride=2, padding=1),
            gn(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            gn(128),
            nn.ReLU(),
        )

        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(256, 64, kernel_size=4, stride=2, padding=1),
            gn(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            gn(64),
            nn.ReLU(),
        )

        self.dec4 = nn.Sequential(
            nn.ConvTranspose2d(128, 32, kernel_size=4, stride=2, padding=1),
            gn(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            gn(32),
            nn.ReLU(),
        )

        # Upsample only: 448 -> 896
        self.dec5 = nn.Sequential(
            nn.ConvTranspose2d(64, 16, kernel_size=4, stride=2, padding=1),
            gn(16),
            nn.ReLU(),
        )

        # Full-res head after concatenating the e0 skip
        self.head = nn.Sequential(
            nn.Conv2d(16 + c0, 16, kernel_size=3, padding=1),
            gn(16),
            nn.ReLU(),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            gn(16),
            nn.ReLU(),
            nn.Conv2d(16, 1, kernel_size=3, padding=1),   # logits
        )

    def forward(self, x, skip_connections):
        e0, e1, e2, e3, e4 = skip_connections
        d1 = self.dec1(x)                                  # 56
        d2 = self.dec2(torch.cat([d1, e4], dim=1))         # 112
        d3 = self.dec3(torch.cat([d2, e3], dim=1))         # 224
        d4 = self.dec4(torch.cat([d3, e2], dim=1))         # 448
        d5 = self.dec5(torch.cat([d4, e1], dim=1))         # 896
        return self.head(torch.cat([d5, e0], dim=1))


class LineInpainter(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = UNetEncoder()

        # Bottleneck attention: global reasoning at coarsest scale (28x28)
        self.bottleneck_attn = BottleneckAttention(
            dim=512, heads=8, num_blocks=6, spatial_size=28
        )

        # dilated context stack at 56x56, before mid attention.
        # Cheaply expands receptive field to span big holes.
        self.mid_dilated = DilatedContextStack(
            channels=256, dilations=(1, 2, 4, 8, 16)
        )

        # Mid attention at 56x56 (each token = 16x16 patch)
        self.mid_attn = BottleneckAttention(
            dim=256, heads=8, num_blocks=3, spatial_size=56
        )

        self.decoder = UNetDecoder()

    def forward(self, x):
        bottleneck, skips = self.encoder(x)
        e0, e1, e2, e3, e4 = skips

        bottleneck = self.bottleneck_attn(bottleneck)

        e4 = self.mid_dilated(e4)      # 56x56
        e4 = self.mid_attn(e4)

        output = self.decoder(bottleneck, [e0, e1, e2, e3, e4])
        return output if self.training else torch.sigmoid(output)



# **LOSS**

In [ ]:
#CODE BLOCK 10 (LOSS)

class LineLoss(nn.Module):
    def __init__(self, hole_weight=3.0, bce_weight=1.0, dice_weight=1.0,
                 gap_dice_weight=0.5, gap_radius=7, smooth=1.0):
        super().__init__()
        self.hole_weight, self.bce_weight = hole_weight, bce_weight
        self.dice_weight, self.gap_dice_weight = dice_weight, gap_dice_weight
        self.gap_radius, self.smooth = gap_radius, smooth
        self.last_components = {}

    @staticmethod
    def _hole_mask(corrupted, target):
        return ((corrupted > 0.9) & (target < 0.5)).float()

    def _dice(self, p, t, region=None):
        if region is not None:
            p, t = p * region, t * region
        dims  = (1, 2, 3)
        inter = (p * t).sum(dims)
        card  = p.sum(dims) + t.sum(dims)
        return 1.0 - ((2 * inter + self.smooth) / (card + self.smooth)).mean()

    def forward(self, pred_logits, target, corrupted):
        pred_logits = pred_logits.float()
        target, corrupted = target.float(), corrupted.float()

        hole = self._hole_mask(corrupted, target)
        w    = 1.0 + hole * (self.hole_weight - 1.0)
        per_px = F.binary_cross_entropy_with_logits(pred_logits, target, reduction='none')
        L_bce  = (per_px * w).mean()

        pred_lines   = torch.sigmoid(-pred_logits)      # == 1 - sigmoid(x), no cancellation
        target_lines = (target < 0.5).float()           # binarized foreground for Dice

        r = self.gap_radius
        gap_region = F.max_pool2d(hole, 2 * r + 1, stride=1, padding=r)

        L_dice     = self._dice(pred_lines, target_lines)
        L_gap_dice = self._dice(pred_lines, target_lines, gap_region)  # 0 if no holes

        total = (self.bce_weight * L_bce + self.dice_weight * L_dice
                 + self.gap_dice_weight * L_gap_dice)

        with torch.no_grad():
            hole_only = (per_px * hole).sum() / hole.sum().clamp(min=1.0)
        self.last_components = dict(
            bce=L_bce.item(), dice=L_dice.item(), gap_dice=L_gap_dice.item(),
            total=total.item(), hole_frac=hole.mean().item(), hole_only_bce=hole_only.item())
        return total



# **METRICS**



In [ ]:
# CODE BLOCK 11 (METRICS)

import numpy as np
import cv2

def _dilate(mask, tol):
    if tol <= 0:
        return mask
    k = np.ones((2 * tol + 1, 2 * tol + 1), np.uint8)
    return cv2.dilate(mask.astype(np.uint8), k).astype(bool)

def _count_regions(img_bool_ink, min_area=20):
    """Number of enclosed white regions. 4-connectivity on background pairs
       correctly with 8-connected ink (diagonal strokes still separate)."""
    bg = (~img_bool_ink).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(bg, connectivity=4)
    # label 0 = ink; drop specks so anti-aliasing noise doesn't inflate the count
    return int(sum(1 for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= min_area))

def line_stats(corrupted, pred, clean, tol=1, min_area=20, regions=True):
    """
    Raw counts for one image. All inputs: HxW arrays in [0,1], ink = < 0.5.
    Counts are summed over images, then turned into scores with summarize().
    """
    c_ink, p_ink, g_ink = corrupted < 0.5, pred < 0.5, clean < 0.5

    added = p_ink & ~c_ink            # ink the model drew
    gap   = g_ink & ~c_ink            # ink that was missing

    good_added = added & _dilate(g_ink, tol)
    found_gap  = gap & _dilate(added, tol)

    out = dict(n_added=int(added.sum()), good_added=int(good_added.sum()),
               n_gap=int(gap.sum()), found_gap=int(found_gap.sum()),
               # did the model keep the ink it was handed?
               n_input_ink=int(c_ink.sum()), kept_ink=int((c_ink & p_ink).sum()))

    if regions:
        r_gt   = _count_regions(g_ink, min_area)
        r_pred = _count_regions(p_ink, min_area)
        r_corr = _count_regions(c_ink, min_area)
        out.update(r_gt=r_gt, r_pred=r_pred, r_corr=r_corr,
                   r_exact=int(r_pred == r_gt),
                   r_lost=max(r_gt - r_corr, 0),                  # regions the holes merged
                   r_recovered=max(min(r_pred, r_gt) - r_corr, 0),
                   n_images=1)
    return out

def summarize(t):
    precision = t['good_added'] / t['n_added'] if t['n_added'] > 0 else 0.0
    recall    = t['found_gap'] / max(t['n_gap'], 1)
    f1        = 2 * precision * recall / max(precision + recall, 1e-8)
    out = dict(precision=precision, recall=recall, f1=f1)

    if 'n_input_ink' in t:
        out['ink_kept'] = t['kept_ink'] / max(t['n_input_ink'], 1)
    if 'r_gt' in t:
        out['region_acc']      = t['r_exact'] / max(t.get('n_images', 1), 1)
        out['region_recovery'] = t['r_recovered'] / max(t['r_lost'], 1)
        out['regions_per_img'] = (t['r_pred'] / max(t.get('n_images', 1), 1),
                                  t['r_gt']   / max(t.get('n_images', 1), 1))
    return out

def add_stats(total, s):
    for k, v in s.items():
        total[k] = total.get(k, 0) + v
    return total

# **TRAIN**

TRAIN UTILITIES 1: Predictions and epoch evaluation

In [ ]:
# CODE BLOCK 12a (train utilities 1)

import os
import numpy as np
import torch
import matplotlib.pyplot as plt

import cv2
from torch.amp import autocast

PREVIEW_DIR = "/content/epoch_previews"
os.makedirs(PREVIEW_DIR, exist_ok=True)
METRIC_TOL = 0   # pixel tolerance for ALL metrics: previews and test set

def error_map(corr, pred_bin, gt):
    """White bg, black = correct ink, green = correctly filled gap,
       red = missed ink, blue = false ink."""
    c_ink, p_ink, g_ink = corr < 0.5, pred_bin < 0.5, gt < 0.5
    rgb = np.ones(corr.shape + (3,), np.float32)
    rgb[p_ink & g_ink]          = (0.0, 0.0, 0.0)
    rgb[p_ink & g_ink & ~c_ink] = (0.0, 0.7, 0.0)
    rgb[g_ink & ~p_ink]         = (0.9, 0.0, 0.0)
    rgb[p_ink & ~g_ink]         = (0.0, 0.3, 1.0)
    gap = g_ink & ~c_ink
    fill_recall = (p_ink & gap).sum() / max(gap.sum(), 1)
    return rgb, fill_recall


@torch.no_grad()
def show_predictions(model, corrupted, clean, tag):
    was_training = model.training
    model.eval()                       # eval mode: forward returns probabilities
    with autocast('cuda', dtype=torch.float16):
        pred = model(corrupted.to(device))

    pred = pred.float().cpu()
    if was_training:
        model.train()
    binar = (pred > 0.5).float()

    n = corrupted.shape[0]
    fig, axes = plt.subplots(n, 5, figsize=(25, 5 * n))
    axes = np.atleast_2d(axes)
    for r in range(n):
        c, p, b, g = (t[r, 0].numpy() for t in (corrupted, pred, binar, clean))
        err, _ = error_map(c, b, g)                       # colors only
        st = line_stats(c, b, g, tol=METRIC_TOL)          # same code as evaluate()
        sm = summarize(st)
        err_title = (f"Errors | fill P {sm['precision']:.0%} R {sm['recall']:.0%}"
                     f" | regions {st['r_pred']}/{st['r_gt']}")
        panels = [(c, "Corrupted"), (p, "Prediction"), (b, "Binarized"),
                  (g, "Ground truth"), (err, err_title)]
        for col, (img, title) in enumerate(panels):
            ax = axes[r, col]
            if img.ndim == 2:
                ax.imshow(img, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            else:
                ax.imshow(img, interpolation="nearest")
            ax.set_title(f"{EXAMPLE_FILES[r]} - {title}" if col == 0 else title)
            ax.axis("off")
    fig.suptitle(str(tag), fontsize=18)
    plt.tight_layout()
    fig.savefig(os.path.join(PREVIEW_DIR, f"{tag}.png"), dpi=100)
    plt.show()
    plt.close(fig)

@torch.no_grad()
def evaluate(model, loader, tol=METRIC_TOL, min_area=20, regions=True, max_batches=None):
    was_training = model.training
    model.eval()
    total = {}
    tp = fp = fn = 0

    for b, (corr, clean, _) in enumerate(loader):
        with autocast('cuda', dtype=torch.float16):
            pred = model(corr.to(device, non_blocking=True))   # eval mode -> probabilities
        pred = pred.float().cpu()

        # global pixel counts (only used for ink_ratio / px_f1)
        p_ink, g_ink = pred < 0.5, clean < 0.5
        tp += (p_ink & g_ink).sum().item()
        fp += (p_ink & ~g_ink).sum().item()
        fn += (~p_ink & g_ink).sum().item()

        # per-image structural stats (CPU / OpenCV)
        for c, p, g in zip(corr[:, 0].numpy(), pred[:, 0].numpy(), clean[:, 0].numpy()):
            add_stats(total, line_stats(c, p, g, tol=tol, min_area=min_area, regions=regions))

        if max_batches and b + 1 >= max_batches:
            break

    if was_training:
        model.train()

    s = summarize(total)
    px_p, px_r = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    out = dict(
        fill_precision = s['precision'],    # added ink that lands on real lines (±tol px)
        fill_recall    = s['recall'],       # erased ink that got redrawn (±tol px)
        fill_f1        = s['f1'],
        ink_kept       = s['ink_kept'],     # given ink the model preserved
        px_f1          = 2 * px_p * px_r / max(px_p + px_r, 1e-9),
        ink_ratio      = (tp + fp) / max(tp + fn, 1),   # >1 = lines too thick
    )
    if regions:
        rp, rg = s['regions_per_img']
        out.update(region_acc      = s['region_acc'],        # images with exact region count
                   region_recovery = s['region_recovery'],   # merged regions re-separated
                   regions_pred    = rp,
                   regions_gt      = rg)
    return out

# TRAIN UTILITIES 2: Curves

In [ ]:
# CODE BLOCK 12b (train utilities b, curves)

LOSS_KEYS = ("total", "bce", "dice", "gap_dice", "hole_only_bce")

def _smooth(y, w=50):
    y = np.asarray(y, float)
    if len(y) < w:
        return y
    return np.convolve(y, np.ones(w) / w, mode="valid")

def plot_curves(loss_log, history, tag, window=50):
    fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))

    # Left: training losses (moving average, log scale so all components fit)
    ax = axes[0]
    for k in LOSS_KEYS:
        y = loss_log.get(k, [])
        if not y:
            continue
        s = _smooth(y, window)
        x = np.arange(len(y) - len(s) + 1, len(y) + 1)   # align each average to its last step
        ax.plot(x, s, label=k)
    ax.set_yscale("log")
    ax.set_xlabel("step"); ax.set_title(f"train losses (moving avg {window})")
    ax.legend(); ax.grid(alpha=.3)

    # Right: test metrics, one point per epoch
    ax = axes[1]
    if history:
        ep = np.arange(1, len(history) + 1)
        for k in ("fill_precision", "fill_recall", "fill_f1", "region_acc", "ink_kept"):
            ax.plot(ep, [h[k] for h in history], marker="o", label=k)
        ax.set_ylim(0, 1); ax.legend()
    else:
        ax.text(0.5, 0.5, "test metrics appear after epoch 1",
                ha="center", va="center", transform=ax.transAxes)
    ax.set_xlabel("epoch"); ax.set_title("test metrics"); ax.grid(alpha=.3)

    fig.suptitle(str(tag), fontsize=14)
    plt.tight_layout()
    fig.savefig(os.path.join(PREVIEW_DIR, f"curves_{tag}.png"), dpi=100)
    plt.show()
    plt.close(fig)

In [ ]:
# CODE BLOCK 13 (TRAIN)

import torch.nn.functional as F
from torch.amp import autocast, GradScaler

train_losses, history = [], []
loss_log = {k: [] for k in LOSS_KEYS}

PREVIEW_EVERY = 500
global_step   = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = LineInpainter().to(device)

epochs = 30

# Lower learning rate for the attention blocks
attn_params = list(model.bottleneck_attn.parameters()) + list(model.mid_attn.parameters())
attn_ids    = {id(p) for p in attn_params}
conv_params = [p for p in model.parameters() if id(p) not in attn_ids]

optimizer = torch.optim.AdamW([
    {'params': conv_params, 'lr': 1e-3},
    {'params': attn_params, 'lr': 3e-4},
], weight_decay=1e-4)

n_opt = sum(p.numel() for g in optimizer.param_groups for p in g['params'])
assert n_opt == sum(p.numel() for p in model.parameters()), "params missing from optimizer"

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs, eta_min=1e-6
)
scaler = GradScaler('cuda')

criterion = LineLoss(
    hole_weight=3.0,
    bce_weight=1.0,
    dice_weight=1.0,
).to(device)

for epoch in range(epochs):

    model.train()

    for i, (corrupted, clean) in enumerate(train_loader):
        corrupted, clean = corrupted.to(device), clean.to(device)

        # Mixed-precision forward pass, loss in float32
        optimizer.zero_grad()
        with autocast('cuda', dtype=torch.float16):
            output = model(corrupted)
        loss = criterion(output.float(), clean, corrupted)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        train_losses.append(loss.item())
        global_step += 1
        for k in LOSS_KEYS:
            loss_log[k].append(criterion.last_components[k])

        if i % 10 == 0:
            c = criterion.last_components
            print(
                f"Epoch [{epoch+1}/{epochs}] Step [{i}/{len(train_loader)}] "
                f"total={c['total']:.4f} bce={c['bce']:.4f} dice={c['dice']:.4f} "
                f"hole_only={c['hole_only_bce']:.4f} hole_frac={c['hole_frac']:.4f}"
            )

        # Periodic preview
        if global_step % PREVIEW_EVERY == 0:
            show_predictions(model, ex_corrupted, ex_clean, f"step_{global_step:06d}")
            plot_curves(loss_log, history, f"step_{global_step:06d}")
    print(f"Epoch [{epoch+1}/{epochs}]")

    m = evaluate(model, test_loader)
    history.append(m)
    print("  TEST " + " ".join(f"{k}={v:.4f}" for k, v in m.items()))
    show_predictions(model, ex_corrupted, ex_clean, f"epoch_{epoch+1:03d}")

    scheduler.step()

print("Done!")

# **SAVE CHECKPOINT**

In [ ]:
# SAVE CHECKPOINT TO LOCAL DISK
import os, torch
from google.colab import files

os.makedirs("/content/checkpoints", exist_ok=True)
epochs_done = len(history)          # one entry per finished epoch (evaluate runs at epoch end)
path = f"/content/checkpoints/inkrepair_ep{epochs_done:03d}.pt"

torch.save(dict(
    model       = model.state_dict(),
    optimizer   = optimizer.state_dict(),   # Adam moments: needed to resume smoothly
    scheduler   = scheduler.state_dict(),   # where the cosine LR schedule is
    scaler      = scaler.state_dict(),      # fp16 loss scale
    history     = history,                  # test metrics per epoch
    loss_log    = loss_log,                 # per-step losses for the curves
    global_step = global_step,
    epochs_done = epochs_done,
), path)

print(f"{path}: {os.path.getsize(path) / 1e6:.0f} MB")
files.download(path)

# ***LOAD CHECKPOINT***

In [ ]:
# CODE BLOCK 13b (RESTORE CHECKPOINT)
# Run after blocks 1-12, instead of block 13's initialisation.
# Restores model, optimizer, scheduler, scaler, history, loss_log, global_step.
import os, glob, torch
from torch.amp import GradScaler

CKPT_SOURCE = "drive"   # "upload" (file dialog), "drive" (gdown link), or "local" (path below)
CKPT_PATH   = None       # "local": a .pt path; None = newest file in /content/checkpoints
# "drive": the Drive share link of the .pt file
CKPT_URL    = "https://drive.google.com/file/d/1li1ZjYNWHQfcrC2lq19JLoKHxM9KuW77/view?usp=drive_link"
EPOCHS      = 30         # total epochs of the schedule (must match block 13's `epochs`)

# 1. get the file
os.makedirs("/content/checkpoints", exist_ok=True)
if CKPT_SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    assert up, "nothing uploaded"
    name = next(iter(up))
    CKPT_PATH = os.path.join("/content/checkpoints", os.path.basename(name))
    os.replace(os.path.abspath(name), CKPT_PATH)
elif CKPT_SOURCE == "drive":
    import gdown
    CKPT_PATH = "/content/checkpoints/restored.pt"
    gdown.download(CKPT_URL, CKPT_PATH, quiet=False, fuzzy=True)
elif CKPT_PATH is None:
    found = sorted(glob.glob("/content/checkpoints/*.pt"))
    assert found, "no .pt files in /content/checkpoints"
    CKPT_PATH = found[-1]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)  # trusted checkpoint
print(f"loaded {CKPT_PATH} ({os.path.getsize(CKPT_PATH) / 1e6:.0f} MB)")

# 2. rebuild everything exactly as in block 13, then load the states
epochs = EPOCHS
model = LineInpainter().to(device)
model.load_state_dict(ckpt["model"])

attn_params = list(model.bottleneck_attn.parameters()) + list(model.mid_attn.parameters())
attn_ids    = {id(p) for p in attn_params}
conv_params = [p for p in model.parameters() if id(p) not in attn_ids]
optimizer = torch.optim.AdamW([
    {'params': conv_params, 'lr': 1e-3},
    {'params': attn_params, 'lr': 3e-4},
], weight_decay=1e-4)
optimizer.load_state_dict(ckpt["optimizer"])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
scheduler.load_state_dict(ckpt["scheduler"])

scaler = GradScaler('cuda')
scaler.load_state_dict(ckpt["scaler"])

criterion = LineLoss(hole_weight=3.0, bce_weight=1.0, dice_weight=1.0).to(device)

# 3. training bookkeeping
history     = ckpt["history"]
loss_log    = ckpt["loss_log"]
global_step = ckpt["global_step"]
start_epoch = ckpt["epochs_done"]
train_losses = list(loss_log.get("total", []))
PREVIEW_EVERY = 500

print(f"restored at epoch {start_epoch}/{epochs}, step {global_step}, "
      f"lr {[round(g['lr'], 6) for g in optimizer.param_groups]}")
if history:
    print("  last TEST " + " ".join(f"{k}={v:.4f}" for k, v in history[-1].items()
                                    if isinstance(v, (int, float))))

# 4. sanity check: the restored model must reproduce the saved test metrics
model.eval()
m = evaluate(model, test_loader, max_batches=25)
print("  re-eval (25 batches) " + " ".join(f"{k}={v:.4f}" for k, v in m.items()))
show_predictions(model, ex_corrupted, ex_clean, f"restored_ep{start_epoch:03d}")
plot_curves(loss_log, history, f"restored_ep{start_epoch:03d}")




# ***FINAL RESULTS***



In [ ]:
# CODE BLOCK 14 (PAPER RESULTS: final evaluation on the frozen test set)
#
# Runs once after training. Needs from earlier blocks: model, device,
# test_loader, test_dataset, EXAMPLE_FILES, line_stats (11), error_map (12a).
#
# Reports, for OUR model and for baselines on the SAME test images:
#   * Input (no restoration)  -> lower bound, shows what the holes cost
#   * Morphological closing   -> classical mask-free gap closing
#   * Ours
#
# Metric families
#   Fill      : precision / recall / F1 of the ink the method ADDS (tol 0 and 2 px)
#   Pixel     : IoU / F1 of all ink, ink kept, ink ratio (>1 = lines too thick)
#   Regions   : exact region count, merged regions re-separated
#   Bucket    : paint-bucket view. Each GT closed region is matched to the best
#               predicted region (IoU). "Bucket acc." = GT regions matched with
#               IoU >= BUCKET_IOU. A "leak" = one predicted region covering
#               >= 50% of two GT regions (the bucket fill would spill over).
#   Gaps      : each erased segment (connected component of GT ink missing in
#               the input) is "closed" if >= GAP_CLOSED of its pixels are redrawn;
#               reported overall and by gap length.
#
# Ratios are pooled over all test images (micro average); 95% CIs come from a
# bootstrap over images. Paired bootstrap p-values compare Ours vs each baseline.
#
# Outputs in RESULTS_DIR: main table (CSV + LaTeX), gap-size table, per-image
# CSV, summary JSON, figures (PDF + PNG), model weights, and a zip of it all.
import os, json, time, shutil, platform
import numpy as np
import pandas as pd
import torch, cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from torch.amp import autocast
from IPython.display import display

# configuration
RESULTS_DIR   = "/content/paper_results"
CHECKPOINT    = None       # path to saved weights; None = use the model in memory
COPY_TO_DRIVE = None       # e.g. "/content/drive/MyDrive/paper_results" (Drive must be mounted)
THRESH        = 0.5        # ink = value < THRESH (same as training metrics)
MIN_AREA      = 20         # ignore white regions smaller than this (anti-aliasing specks)
TOLS          = (0, 2)     # pixel tolerances for fill precision / recall
BUCKET_IOU    = 0.9        # GT region is correctly paintable if matched with IoU >= this
CLOSE_KS      = (5, 9)     # kernel sizes (px) of the morphological-closing baselines
GAP_CLOSED    = 0.8        # an erased segment is closed if >= 80% of its pixels are redrawn
GAP_MIN_AREA  = 4          # ignore erased "segments" smaller than this (edge specks)
GAP_BINS      = (0, 8, 16, 32, 64, np.inf)   # gap length (px, longest bbox side)
N_BOOT        = 1000
SEED          = 0
OURS          = "Ours"
os.makedirs(RESULTS_DIR, exist_ok=True)


# helpers
def div(a, b):
    """Elementwise a / b, NaN where b == 0 (undefined, shown as '—')."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    return np.where(b > 0, a / np.where(b > 0, b, 1.0), np.nan)

def f1(p, r):
    s = p + r
    return np.where(s > 0, 2 * p * r / np.where(s > 0, s, 1.0), 0.0) * np.where(np.isnan(s), np.nan, 1.0)

def closing_baseline(corr, k):
    """Classical gap closing: morphological closing of the ink mask."""
    ink = (corr < THRESH).astype(np.uint8)
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    return 1.0 - cv2.morphologyEx(ink, cv2.MORPH_CLOSE, ker).astype(np.float32)

def _bg_labels(ink):
    # 4-connected white regions (pairs with 8-connected ink); label 0 = ink pixels
    n, lab, st, _ = cv2.connectedComponentsWithStats((~ink).astype(np.uint8), connectivity=4)
    return n, lab, st[:, cv2.CC_STAT_AREA]

def region_stats(g_ink, p_ink):
    """Paint-bucket metrics: best-IoU matching of GT regions and leak count."""
    ng, lg, ag = _bg_labels(g_ink)
    npr, lp, ap = _bg_labels(p_ink)
    codes, inter = np.unique(lg.astype(np.int64).ravel() * npr + lp.ravel(), return_counts=True)
    gi, pj = codes // npr, codes % npr
    ok = (gi > 0) & (pj > 0) & (ag[gi] >= MIN_AREA)
    gi, pj, inter = gi[ok], pj[ok], inter[ok]

    valid_g = np.zeros(ng, bool); valid_g[1:] = ag[1:] >= MIN_AREA
    best = np.zeros(ng)
    if len(gi):
        iou = inter / (ag[gi] + ap[pj] - inter)
        np.maximum.at(best, gi, iou)
    best = best[valid_g]

    # leak: a predicted region that holds the majority of >= 2 GT regions
    maj = inter >= 0.5 * ag[gi]
    per_pred = np.bincount(pj[maj], minlength=npr) if maj.any() else np.zeros(1, int)
    n_leaks = int(np.maximum(per_pred - 1, 0).sum())
    return dict(n_gt_regions=int(valid_g.sum()),
                bucket_ok=int((best >= BUCKET_IOU).sum()),
                sum_best_iou=float(best.sum()),
                n_leaks=n_leaks, leak_free=int(n_leaks == 0))

def gap_components(c_ink, g_ink):
    """Erased segments of the test image (method independent)."""
    gap = (g_ink & ~c_ink).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(gap, connectivity=8)
    area = st[:, cv2.CC_STAT_AREA]
    length = np.maximum(st[:, cv2.CC_STAT_WIDTH], st[:, cv2.CC_STAT_HEIGHT])
    keep = np.zeros(n, bool); keep[1:] = area[1:] >= GAP_MIN_AREA
    return n, lab, area, length, keep

def gap_recall(p_ink, comps):
    n, lab, area, length, keep = comps
    found = np.bincount(lab.ravel(), weights=p_ink.ravel().astype(float), minlength=n)
    rec = found[keep] / area[keep]
    return length[keep], rec


# metric definitions (computed from pooled column sums)
# (name, fn(S) -> value, direction: up / down / one, format: pct / num)
def _P(S, t): return div(S[f"good_added_t{t}"], S[f"n_added_t{t}"])
def _R(S, t): return div(S[f"found_gap_t{t}"], S[f"n_gap_t{t}"])

METRICS = []
for t in TOLS:
    METRICS += [
        (f"Fill P (τ={t})",  lambda S, t=t: _P(S, t), "up", "pct"),
        (f"Fill R (τ={t})",  lambda S, t=t: _R(S, t), "up", "pct"),
        (f"Fill F1 (τ={t})", lambda S, t=t: f1(_P(S, t), _R(S, t)), "up", "pct"),
    ]
METRICS += [
    ("Pixel IoU",         lambda S: div(S["tp"], S["tp"] + S["fp"] + S["fn"]), "up", "pct"),
    ("Pixel F1",          lambda S: div(2 * S["tp"], 2 * S["tp"] + S["fp"] + S["fn"]), "up", "pct"),
    ("Ink kept",          lambda S: div(S["kept_ink"], S["n_input_ink"]), "up", "pct"),
    ("Ink ratio",         lambda S: div(S["tp"] + S["fp"], S["tp"] + S["fn"]), "one", "num"),
    ("Gap closure",       lambda S: div(S["gaps_closed"], S["n_gaps"]), "up", "pct"),
    ("Region count acc.", lambda S: div(S["r_exact"], S["n_images"]), "up", "pct"),
    ("Region recovery",   lambda S: div(S["r_recovered"], S["r_lost"]), "up", "pct"),
    (f"Bucket acc. (IoU≥{BUCKET_IOU})", lambda S: div(S["bucket_ok"], S["n_gt_regions"]), "up", "pct"),
    ("Mean region IoU",   lambda S: div(S["sum_best_iou"], S["n_gt_regions"]), "up", "pct"),
    ("Leaks / image",     lambda S: div(S["n_leaks"], S["n_images"]), "down", "num"),
    ("Leak-free images",  lambda S: div(S["leak_free"], S["n_images"]), "up", "pct"),
]
KEY_METRICS = [f"Fill F1 (τ={TOLS[0]})", "Gap closure",
               f"Bucket acc. (IoU≥{BUCKET_IOU})", "Leak-free images"]


def per_image_record(c, p, g, comps):
    c_ink, p_ink, g_ink = c < THRESH, p < THRESH, g < THRESH
    rec = dict(tp=int((p_ink & g_ink).sum()), fp=int((p_ink & ~g_ink).sum()),
               fn=int((~p_ink & g_ink).sum()))
    for t in TOLS:
        s = line_stats(c, p, g, tol=t, min_area=MIN_AREA, regions=(t == TOLS[0]))
        for k in ("n_added", "good_added", "n_gap", "found_gap"):
            rec[f"{k}_t{t}"] = s[k]
        if t == TOLS[0]:
            rec.update({k: s[k] for k in ("n_input_ink", "kept_ink", "r_gt", "r_pred",
                                          "r_corr", "r_exact", "r_lost", "r_recovered",
                                          "n_images")})
    rec.update(region_stats(g_ink, p_ink))
    lengths, rec_g = gap_recall(p_ink, comps)
    rec.update(n_gaps=len(rec_g), gaps_closed=int((rec_g >= GAP_CLOSED).sum()))
    return rec, lengths, rec_g


def evaluate_all(methods_fn):
    """methods_fn(corr_batch) -> {method: (B, H, W) float arrays}."""
    rows, gaps = [], []
    for corr, clean, names in test_loader:
        preds = methods_fn(corr)
        for b, fname in enumerate(names):
            c, g = corr[b, 0].numpy(), clean[b, 0].numpy()
            comps = gap_components(c < THRESH, g < THRESH)
            hole = float(((c > 0.9) & (g < 0.5)).mean())
            for m, P in preds.items():
                rec, lengths, rec_g = per_image_record(c, P[b], g, comps)
                rows.append(dict(method=m, file=fname, hole_frac=hole, **rec))
                gaps.append(pd.DataFrame(dict(method=m, file=fname, length=lengths, recall=rec_g)))
    return pd.DataFrame(rows), pd.concat(gaps, ignore_index=True)


def aggregate(df, methods, rng):
    """Point estimates + bootstrap CIs; the same resamples for every method (paired)."""
    files = sorted(df["file"].unique())
    n = len(files)
    idx = rng.integers(0, n, (N_BOOT, n))
    cols = [c for c in df.columns if c not in ("method", "file", "hole_frac")]
    point, boot = {}, {}
    for m in methods:
        X = df[df["method"] == m].set_index("file").loc[files, cols]
        S_full = {c: X[c].to_numpy(float).sum() for c in cols}
        S_boot = {c: X[c].to_numpy(float)[idx].sum(1) for c in cols}
        point[m] = {name: float(fn(S_full)) for name, fn, *_ in METRICS}
        boot[m]  = {name: np.asarray(fn(S_boot), float) for name, fn, *_ in METRICS}
    return point, boot, n


def fmt(v, kind):
    if v is None or np.isnan(v):
        return "—"
    return f"{100 * v:.1f}" if kind == "pct" else f"{v:.3f}"


def latex_table(point, boot, methods, metric_names, caption, label):
    spec = {name: (d, k) for name, _, d, k in METRICS}
    arrows = {"up": r"$\uparrow$", "down": r"$\downarrow$", "one": r"$\to 1$"}
    def best_of(name):
        d, _ = spec[name]
        vals = {m: point[m][name] for m in methods if not np.isnan(point[m][name])}
        if not vals:
            return None
        key = {"up": lambda m: vals[m], "down": lambda m: -vals[m],
               "one": lambda m: -abs(vals[m] - 1)}[d]
        return max(vals, key=key)
    def tex_name(n):   # no backslashes inside f-string expressions (Python < 3.12)
        return n.replace("τ", r"$\tau$").replace("≥", r"$\geq$")
    head = " & ".join(["Method"] + [tex_name(n) + " " + arrows[spec[n][0]] for n in metric_names])
    lines = [r"\begin{table}[t]", r"\centering", r"\small",
             r"\resizebox{\linewidth}{!}{%",
             r"\begin{tabular}{l" + "c" * len(metric_names) + "}", r"\toprule",
             head + r" \\", r"\midrule"]
    # bold every method that ties with the best (as printed); no bold if all tie
    bold = {}
    for n in metric_names:
        b = best_of(n)
        shown = {m: fmt(point[m][n], spec[n][1]) for m in methods}
        winners = {m for m in methods if b is not None and shown[m] == shown[b]}
        bold[n] = winners if len(winners) < len(methods) else set()
    for m in methods:
        cells = [m]
        for n in metric_names:
            d, k = spec[n]
            v = point[m][n]
            lo, hi = np.nanpercentile(boot[m][n], [2.5, 97.5]) if not np.isnan(v) else (np.nan, np.nan)
            s = fmt(v, k)
            if s == "—":
                cells.append("--")
                continue
            if m in bold[n]:
                s = r"\textbf{" + s + "}"
            hw = (hi - lo) / 2 * (100 if k == "pct" else 1)
            s += r"{\scriptsize$\pm$" + (f"{hw:.1f}" if k == "pct" else f"{hw:.3f}") + "}"
            cells.append(s)
        lines.append(" & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}}",
              r"\caption{" + caption + "}", r"\label{" + label + "}", r"\end{table}"]
    return "\n".join(lines)


def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(RESULTS_DIR, f"{name}.{ext}"), dpi=300, bbox_inches="tight")


# Run
assert all(v in globals() for v in ("model", "test_loader", "line_stats", "error_map")), \
    "run the data, model, metrics and training blocks first"
t_start = time.perf_counter()

# 1. model
if CHECKPOINT:
    sd = torch.load(CHECKPOINT, map_location=device)
    model.load_state_dict(sd.get("model", sd) if isinstance(sd, dict) else sd)
    print(f"loaded weights from {CHECKPOINT}")
model.eval()                                    # forward returns probabilities
torch.save(model.state_dict(), os.path.join(RESULTS_DIR, "model_weights.pt"))

@torch.no_grad()
def run_model(corr):
    with autocast("cuda", dtype=torch.float16):
        return model(corr.to(device, non_blocking=True)).float().cpu()[:, 0].numpy()

def methods_fn(corr):
    c = corr[:, 0].numpy()
    out = {"Input (no restoration)": c}
    for k in CLOSE_KS:
        out[f"Closing (k={k})"] = np.stack([closing_baseline(x, k) for x in c])
    out[OURS] = run_model(corr)
    return out

METHODS = ["Input (no restoration)"] + [f"Closing (k={k})" for k in CLOSE_KS] + [OURS]

# 2. evaluate every method on every test image
print(f"evaluating {len(test_dataset)} test images × {len(METHODS)} methods ...")
df, gaps_df = evaluate_all(methods_fn)
df.to_csv(os.path.join(RESULTS_DIR, "per_image_metrics.csv"), index=False)

rng = np.random.default_rng(SEED)
point, boot, n_test = aggregate(df, METHODS, rng)

# 3. main table
rows = []
for m in METHODS:
    row = {"Method": m}
    for name, _, d, k in METRICS:
        v = point[m][name]
        if np.isnan(v):
            row[name] = "—"
        else:
            lo, hi = np.nanpercentile(boot[m][name], [2.5, 97.5])
            row[name] = f"{fmt(v, k)} [{fmt(lo, k)}, {fmt(hi, k)}]"
    rows.append(row)
table = pd.DataFrame(rows).set_index("Method")
table_num = pd.DataFrame({m: point[m] for m in METHODS}).T
table_num.to_csv(os.path.join(RESULTS_DIR, "main_table_values.csv"))
table.to_csv(os.path.join(RESULTS_DIR, "main_table_with_ci.csv"))
print("\n=== MAIN RESULTS (% unless noted; [95% bootstrap CI]) ===")
display(table.T)

main_cols = [f"Fill P (τ={TOLS[0]})", f"Fill R (τ={TOLS[0]})", f"Fill F1 (τ={TOLS[0]})",
             f"Fill F1 (τ={TOLS[-1]})", "Pixel F1", "Ink ratio", "Gap closure",
             f"Bucket acc. (IoU≥{BUCKET_IOU})", "Leaks / image", "Leak-free images"]
tex = latex_table(point, boot, METHODS, main_cols,
                  caption=(f"Line restoration on the held-out test set ({n_test} images, "
                           f"{df[df.method == OURS].n_gaps.sum()} erased segments). "
                           "Percentages unless noted; $\\pm$ is half the 95\\% bootstrap CI. "
                           f"$\\tau$: pixel tolerance. Bucket acc.: GT regions recovered with "
                           f"IoU$\\geq${BUCKET_IOU}. Leak: one predicted region spanning two GT regions."),
                  label="tab:main_results")
# all metrics, transposed (metrics as rows) so it fits a page
arrows_t = {"up": r"$\uparrow$", "down": r"$\downarrow$", "one": r"$\to 1$"}
body = []
for name, _, d, k in METRICS:
    cells = [name.replace("τ", r"$\tau$").replace("≥", r"$\geq$") + " " + arrows_t[d]]
    for m in METHODS:
        v = point[m][name]
        cells.append("--" if np.isnan(v) else fmt(v, k))
    body.append(" & ".join(cells) + r" \\")
tex_full = "\n".join(
    [r"\begin{table}[t]", r"\centering", r"\small",
     r"\begin{tabular}{l" + "c" * len(METHODS) + "}", r"\toprule",
     " & ".join(["Metric"] + METHODS) + r" \\", r"\midrule", *body,
     r"\bottomrule", r"\end{tabular}",
     r"\caption{All metrics on the test set (percentages unless noted; 95\% CIs in "
     r"\texttt{main\_table\_with\_ci.csv}).}", r"\label{tab:all_results}", r"\end{table}"])
with open(os.path.join(RESULTS_DIR, "table_main.tex"), "w") as fh:
    fh.write(tex)
with open(os.path.join(RESULTS_DIR, "table_all.tex"), "w") as fh:
    fh.write(tex_full)

# 4. paired significance: Ours vs each baseline
sig = []
for m in METHODS[:-1]:
    for name in KEY_METRICS:
        d = boot[OURS][name] - boot[m][name]
        d = d[~np.isnan(d)]
        if len(d) == 0:
            continue
        sig.append(dict(baseline=m, metric=name,
                        diff=point[OURS][name] - point[m][name],
                        ci_lo=np.percentile(d, 2.5), ci_hi=np.percentile(d, 97.5),
                        p_one_sided=max((d <= 0).mean(), 1 / N_BOOT)))
sig_df = pd.DataFrame(sig)
sig_df.to_csv(os.path.join(RESULTS_DIR, "paired_bootstrap_vs_baselines.csv"), index=False)
print("\n=== Ours − baseline (paired bootstrap; p = P[diff ≤ 0]) ===")
display(sig_df.round(4))

# 5. by gap length
labels = [f"<{GAP_BINS[1]}"] + [f"{a}–{b}" for a, b in zip(GAP_BINS[1:-2], GAP_BINS[2:-1])] \
         + [f"≥{GAP_BINS[-2]}"]
gaps_df["bin"] = pd.cut(gaps_df["length"], bins=list(GAP_BINS), labels=labels, right=False)
gaps_df["closed"] = gaps_df["recall"] >= GAP_CLOSED
by_len = (gaps_df.groupby(["method", "bin"], observed=False)
                 .agg(n=("closed", "size"), closure=("closed", "mean"),
                      mean_recall=("recall", "mean"))
                 .reset_index())
by_len_wide = by_len.pivot(index="bin", columns="method", values="closure")[METHODS] * 100
by_len_wide.insert(0, "# gaps", by_len[by_len.method == OURS].set_index("bin")["n"])
by_len_wide.to_csv(os.path.join(RESULTS_DIR, "gap_closure_by_length.csv"))
print(f"\n=== Gap closure (%) by gap length in px (closed = ≥{GAP_CLOSED:.0%} redrawn) ===")
display(by_len_wide.round(1))
tex_len = by_len_wide[by_len_wide["# gaps"] > 0].rename(columns={"# gaps": r"\# gaps"})
tex_len.index = [str(s).replace("≥", r"$\geq$").replace("<", "$<$").replace("–", "--")
                 for s in tex_len.index]
tex_len.index.name = "Gap length (px)"
tex_len.columns.name = None
closed_pct = f"{GAP_CLOSED:.0%}".replace("%", r"\%")
with open(os.path.join(RESULTS_DIR, "table_gap_length.tex"), "w") as fh:
    fh.write(tex_len.to_latex(
        caption=(r"Gap closure rate (\%) by erased-segment length (longest side of its "
                 r"bounding box, px). A gap is closed if $\geq$" + closed_pct
                 + " of its pixels are redrawn."),
        label="tab:gap_length", na_rep="--", float_format="%.1f", escape=False))

fig, ax = plt.subplots(figsize=(5.5, 3.6))
for m in METHODS:
    ax.plot(labels, by_len_wide[m].values, marker="o", label=m,
            lw=2.5 if m == OURS else 1.2)
ax.set_xlabel("gap length (px)"); ax.set_ylabel("gaps closed (%)")
ax.set_ylim(0, 100); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); savefig(fig, "fig_gap_closure_by_length"); plt.show(); plt.close(fig)

# 6. per-image distributions
t0 = TOLS[0]
df["fill_f1"] = f1(div(df[f"good_added_t{t0}"], df[f"n_added_t{t0}"]),
                   div(df[f"found_gap_t{t0}"], df[f"n_gap_t{t0}"]))
df["bucket_acc"] = div(df["bucket_ok"], df["n_gt_regions"])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, col, title in ((axes[0], "fill_f1", f"per-image fill F1 (τ={t0})"),
                       (axes[1], "bucket_acc", f"per-image bucket acc. (IoU≥{BUCKET_IOU})")):
    data = [df[(df.method == m)][col].dropna().values for m in METHODS]
    ax.boxplot(data, showfliers=False)
    ax.set_xticks(range(1, len(METHODS) + 1))
    ax.set_xticklabels([m.replace(" (", "\n(") for m in METHODS], fontsize=8)
    ax.set_title(title); ax.set_ylim(0, 1.02); ax.grid(alpha=.3, axis="y")
plt.tight_layout(); savefig(fig, "fig_per_image_boxplots"); plt.show(); plt.close(fig)

# 7. qualitative figures
@torch.no_grad()
def predict_file(f):
    corr, clean, _ = test_dataset[test_dataset.files.index(f)]
    return corr[0].numpy(), run_model(corr[None])[0], clean[0].numpy()

def zoom_box(c, g, size=224):
    gap = ((g < THRESH) & ~(c < THRESH)).astype(np.uint8)
    n, _, st, cen = cv2.connectedComponentsWithStats(gap, connectivity=8)
    H, W = g.shape
    cx, cy = (W / 2, H / 2) if n <= 1 else cen[1 + np.argmax(st[1:, cv2.CC_STAT_AREA])]
    y0 = int(np.clip(cy - size / 2, 0, H - size)); x0 = int(np.clip(cx - size / 2, 0, W - size))
    return np.s_[y0:y0 + size, x0:x0 + size]

k_show = max(CLOSE_KS)
legend = [Patch(color=(0, 0, 0), label="correct ink"),
          Patch(color=(0, .7, 0), label="correctly filled gap"),
          Patch(color=(.9, 0, 0), label="missed ink"),
          Patch(color=(0, .3, 1), label="false ink")]

def qual_figure(files, name, zoom=False):
    cols = ["Input", f"Closing (k={k_show})", OURS, "Ground truth", "Error map (ours)"]
    fig, axes = plt.subplots(len(files), len(cols), figsize=(3.2 * len(cols), 3.3 * len(files)))
    axes = np.atleast_2d(axes)
    for r, f in enumerate(files):
        c, p, g = predict_file(f)
        pb = (p >= THRESH).astype(np.float32)
        err, _ = error_map(c, pb, g)
        imgs = [c, closing_baseline(c, k_show), pb, g, err]
        sl = zoom_box(c, g) if zoom else np.s_[:, :]
        for col, (img, title) in enumerate(zip(imgs, cols)):
            ax = axes[r, col]
            ax.imshow(img[sl], cmap=None if img.ndim == 3 else "gray",
                      vmin=0, vmax=1, interpolation="nearest")
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(title, fontsize=11)
        axes[r, 0].set_ylabel(f, fontsize=8)
    fig.legend(handles=legend, loc="lower center", ncol=4, fontsize=9,
               bbox_to_anchor=(0.5, -0.01))
    plt.tight_layout(rect=(0, 0.03, 1, 1)); savefig(fig, name); plt.show(); plt.close(fig)

qual_figure(EXAMPLE_FILES, "fig_qualitative")
qual_figure(EXAMPLE_FILES, "fig_qualitative_zoom", zoom=True)

ours_df = df[df.method == OURS]
worst = (ours_df[ours_df[f"n_gap_t{t0}"] >= 200]
         .sort_values(["fill_f1", "n_leaks"], ascending=[True, False])["file"].head(3).tolist())
if worst:
    qual_figure(worst, "fig_failure_cases", zoom=True)

# 8. cost: parameters, inference time, memory
n_params = sum(p.numel() for p in model.parameters())
timing = {}
if device.type == "cuda":
    x = next(iter(test_loader))[0][:1].to(device)
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(), autocast("cuda", dtype=torch.float16):
        for _ in range(10):
            model(x)
        torch.cuda.synchronize(); t = time.perf_counter()
        for _ in range(50):
            model(x)
        torch.cuda.synchronize()
    timing = dict(ms_per_image=1000 * (time.perf_counter() - t) / 50,
                  peak_mem_gb=torch.cuda.max_memory_allocated() / 1e9,
                  gpu=torch.cuda.get_device_name(0))

# 9. summary JSON + text
summary = dict(
    n_test_images=n_test,
    n_erased_segments=int(ours_df.n_gaps.sum()),
    mean_hole_fraction=float(ours_df.hole_frac.mean()),
    image_size=IMAGE_SIZE, params_M=n_params / 1e6, **timing,
    config=dict(threshold=THRESH, min_region_area=MIN_AREA, tolerances=TOLS,
                bucket_iou=BUCKET_IOU, gap_closed=GAP_CLOSED, gap_min_area=GAP_MIN_AREA,
                closing_kernels=CLOSE_KS, n_boot=N_BOOT, seed=SEED),
    env=dict(torch=torch.__version__, python=platform.python_version()),
    results={m: {k: (None if np.isnan(v) else v) for k, v in point[m].items()} for m in METHODS},
)
with open(os.path.join(RESULTS_DIR, "summary.json"), "w") as fh:
    json.dump(summary, fh, indent=2, default=float)

o = point[OURS]
best_base = max(METHODS[1:-1], key=lambda m: np.nan_to_num(point[m][KEY_METRICS[0]]))
print("\n=== SUMMARY ===")
print(f"test set: {n_test} images ({IMAGE_SIZE}²), {summary['n_erased_segments']} erased "
      f"segments, mean hole fraction {summary['mean_hole_fraction']:.3%}")
print(f"model: {n_params / 1e6:.2f} M params"
      + (f", {timing['ms_per_image']:.1f} ms/image, {timing['peak_mem_gb']:.2f} GB on "
         f"{timing['gpu']}" if timing else ""))
for name in KEY_METRICS + ["Pixel F1", "Ink ratio"]:
    k = next(kk for nn, _, _, kk in METRICS if nn == name)
    print(f"  {name:28s} ours {fmt(o[name], k):>7s} | input "
          f"{fmt(point[METHODS[0]][name], k):>7s} | {best_base} {fmt(point[best_base][name], k):>7s}")

# 10. package
zip_path = shutil.make_archive(RESULTS_DIR, "zip", RESULTS_DIR)
if COPY_TO_DRIVE:
    shutil.copytree(RESULTS_DIR, COPY_TO_DRIVE, dirs_exist_ok=True)
    print(f"copied to {COPY_TO_DRIVE}")
print(f"\nall results in {RESULTS_DIR}  (zip: {zip_path})  "
      f"[{time.perf_counter() - t_start:.0f} s]")
# from google.colab import files; files.download(zip_path)


In [ ]:
# DOWNLOAD PAPER RESULTS
import os, shutil, zipfile
from google.colab import files

RESULTS_DIR   = "/content/paper_results"
INCLUDE_MODEL = False    # model_weights.pt is large; you already have the checkpoint

zip_path = "/content/paper_results.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, fnames in os.walk(RESULTS_DIR):
        for f in sorted(fnames):
            if f == "model_weights.pt" and not INCLUDE_MODEL:
                continue
            full = os.path.join(root, f)
            zf.write(full, os.path.join("paper_results", os.path.relpath(full, RESULTS_DIR)))
            print(f"  {os.path.relpath(full, RESULTS_DIR):45s} {os.path.getsize(full) / 1e6:7.2f} MB")

print(f"\n{zip_path}: {os.path.getsize(zip_path) / 1e6:.1f} MB")
files.download(zip_path)

# ***EXAMPLE IMAGE***

In [ ]:
# CODE BLOCK 15 (INTRO FIGURES: one example per PDF)
#
# For each file in INTRO_FILES, saves a 1x4 figure:
#   corrupted input | prediction (binarized) | ground truth | error map
# as its own PDF, then zips all of them and downloads the zip.
#
# Needs from earlier blocks: model, device, test_dataset (4),
# line_stats / summarize (11), error_map (12a).
# Independent of CODE BLOCK 14 (does not use or change any of its variables).
import os, shutil
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from torch.amp import autocast

# configuration
INTRO_FILES  = [f"pair_{i:05d}.png" for i in range(0, 100, 10)]   # 00000, 00010, ..., 00090
INTRO_DIR    = "/content/intro_figures"
INTRO_THRESH = 0.5      # ink = value < INTRO_THRESH (same as training/test metrics)
SHOW_TITLES  = True     # panel titles above each image
SHOW_LEGEND  = True     # error-map color legend under the figure
SHOW_SCORES  = False    # add fill precision / recall to the error-map title
FIG_WIDTH_IN = 12       # inches; height follows from the 4 square panels

assert all(v in globals() for v in ("model", "device", "test_dataset", "error_map", "line_stats")), \
    "run the data, model, metrics and train-utility blocks (and train / restore the model) first"

shutil.rmtree(INTRO_DIR, ignore_errors=True)       # no stale PDFs in the zip
os.makedirs(INTRO_DIR, exist_ok=True)

ERROR_LEGEND = [Patch(facecolor=(0, 0, 0),    edgecolor="0.5", label="correct ink"),
                Patch(facecolor=(0, .7, 0),   edgecolor="0.5", label="correctly filled gap"),
                Patch(facecolor=(.9, 0, 0),   edgecolor="0.5", label="missed ink"),
                Patch(facecolor=(0, .3, 1),   edgecolor="0.5", label="false ink")]


# helpers
@torch.no_grad()
def intro_predict(corr):
    """corr: (1, H, W) tensor -> (H, W) probability map (eval mode returns sigmoid)."""
    was_training = model.training
    model.eval()
    with autocast("cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
        p = model(corr[None].to(device)).float().cpu()[0, 0].numpy()
    if was_training:
        model.train()
    return p


def intro_figure(fname):
    corr, clean, _ = test_dataset[test_dataset.files.index(fname)]
    c, g = corr[0].numpy(), clean[0].numpy()
    p    = intro_predict(corr)
    pb   = (p >= INTRO_THRESH).astype(np.float32)          # 0 = ink, 1 = paper
    err, _ = error_map(c, pb, g)

    err_title = "Error map"
    if SHOW_SCORES:
        sm = summarize(line_stats(c, pb, g, tol=0, regions=False))
        err_title += f" (P {sm['precision']:.0%}, R {sm['recall']:.0%})"

    panels = [(c,   "Corrupted input"),
              (pb,  "Prediction"),
              (g,   "Ground truth"),
              (err, err_title)]

    with plt.rc_context({"pdf.fonttype": 42, "ps.fonttype": 42,   # editable TrueType text
                         "font.size": 11}):
        fig, axes = plt.subplots(1, 4, figsize=(FIG_WIDTH_IN, FIG_WIDTH_IN / 4 + 0.6))
        for ax, (img, title) in zip(axes, panels):
            # interpolation="none" embeds the original pixels in the PDF (no resampling)
            ax.imshow(img, cmap=None if img.ndim == 3 else "gray",
                      vmin=0, vmax=1, interpolation="none")
            ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_linewidth(0.6); s.set_color("0.4")
            if SHOW_TITLES:
                ax.set_title(title)
        if SHOW_LEGEND:
            fig.legend(handles=ERROR_LEGEND, loc="lower center", ncol=4,
                       frameon=False, fontsize=10, bbox_to_anchor=(0.5, 0.0))
            plt.tight_layout(rect=(0, 0.08, 1, 1))
        else:
            plt.tight_layout()

        out = os.path.join(INTRO_DIR, os.path.splitext(fname)[0] + "_intro.pdf")
        fig.savefig(out, bbox_inches="tight", pad_inches=0.02)
        plt.show()
        plt.close(fig)
    return out


# run
available = set(test_dataset.files)
missing   = [f for f in INTRO_FILES if f not in available]
if missing:
    print(f"WARNING: not in the test set, skipped: {missing}")

saved = []
for f in INTRO_FILES:
    if f in available:
        saved.append(intro_figure(f))
        print(f"saved {saved[-1]}")

# download all PDFs as one zip
assert saved, "no figures were generated"
zip_path = shutil.make_archive(INTRO_DIR, "zip", INTRO_DIR)
print(f"\n{len(saved)} PDFs -> {zip_path}")
from google.colab import files
files.download(zip_path)